# Mamba SOH — LFP chemistry variant (GH-67 Mức 2)

Retrain kiến trúc `MambaSOHPredictor` (window=30, d_model=64, d_state=16) trên dataset
**Severson et al. 2019** (Nature Energy) — LFP/graphite A123 APR18650M1A, 1.1 Ah — để có bộ
artifact riêng cho chemistry LFP. **KHÔNG đụng** model NASA/NMC production (`soh_mamba_v1.6.pth`).

---

## Checklist trước khi Run All

| # | Việc | Ghi chú |
|---|------|---------|
| 1 | Settings → Accelerator → **GPU T4 x2** | KHÔNG chọn P100 — PyTorch Kaggle đã bỏ sm_60 |
| 2 | **+ Add Data** → dataset chứa `.mat` Severson | Batch1/2/3(/4) từ https://data.matr.io/1/ |
| 3 | Add-ons → Secrets → `GITHUB_TOKEN` | GitHub PAT (nếu repo private) |
| 4 | **`git push` code mới lên GitHub TRƯỚC** | Cell 3 sẽ kiểm tra và dừng nếu thiếu |

> ⚠️ **Mục 4 là chỗ dễ mất thời gian nhất.** Notebook clone code từ GitHub remote — sửa file
> trên máy local mà chưa push thì Kaggle **không thấy**. Lần chạy 2026-07-25 mất ~11 giờ và ra
> kết quả y hệt lần trước vì lý do này. Cell 3 giờ tự chặn trường hợp đó.

## Lịch sử kết quả

| Lần | Cấu hình | Test MAE | Test RMSE | Đạt target |
|-----|----------|----------|-----------|------------|
| 1 | `--epochs 5`, `CYCLE_COUNT_NORM=200` | 2.5856% | 3.4745% | ❌ |
| 2 | `--epochs 5`, `CYCLE_COUNT_NORM=2300` | 1.9213% | 2.7627% | ✅ (che lỗi EOL) |
| 3 | + pha xả + lọc outlier + `--balance-bands`, 50 epoch | **1.4365%** | **1.8767%** | ✅ |
| 4 | + `time` giây (#7) + chặn nhiễu feature (#8) + `--swa` | ? | ? | — |

Target chính thức: **MAE < 2.0%** · **RMSE < 3.0%** — đã đạt từ lần 3.
Tham chiếu NASA v1.6 cùng kiến trúc: 1.34% / 1.84%. **Mục tiêu mới: cả hai < 1.0%.**

> RMSE mới là ràng buộc thật: RMSE ≥ MAE luôn, tỉ lệ hiện tại 1.31 → muốn RMSE < 1.0 thì MAE
> phải xuống ~0.77%, tức cải thiện **47%**. Chưa model nào của dự án (kể cả NASA) xuống dưới 1%.

**Vì sao kỳ vọng nhiều ở fix #7:** mọi window cắt từ cùng 1 chu kỳ xả mang **chung 1 nhãn SOH**.
Lát đầu xả (3.4 V) và lát cuối xả (2.9 V) trông khác hẳn nhau nhưng phải cho cùng con số. Model
chỉ phân biệt được nếu biết **đang ở đoạn nào của quá trình xả** — tín hiệu đó chính là
`soc_percent`, mà nó đang chết vì bug #7. Fix này tấn công trực diện nguồn sai số lớn nhất.

## 🚨 Bug #7 — cột `time` của Severson là PHÚT, không phải giây

Đo trên artifact lần 3: scaler fit `time` trên **[0.000, 24.130]**. Xả 4C của cell 1.1 Ah kéo
dài ~15 phút → 24.13 là **phút**. NASA và payload BE gửi đều dùng **giây**. Hai hậu quả:

1. **Kênh `soc_percent` chết lúc train** — `compute_soc_percent()` làm `time / 3600` (giả định
   giây). Cho ăn phút → đếm thiếu 60×: window 12 phút cho `soc 100.00 → 98.79`, đúng phải là
   `100.00 → 27.27`. Model mất 1 trong 6 kênh input.
2. **Lệch phân bố lúc inference** — BE gửi giây, scaler fit trên phút: `60 s → 2.49`,
   `300 s → 12.43`, `720 s → 29.84`, trong khi train chỉ thấy `[0, 1]`. Không lỗi nào được raise.

Đã fix bằng `--time-unit minutes` (mặc định) quy về giây ngay lúc parse.

## 🚨 Bug #8 — 7 đặc trưng đang khuếch đại nhiễu làm tròn 15.000–93.000×

Đo trên `feature_scaler_lfp.pkl` lần 3: 7/57 đặc trưng có `var` ở mức **1e-10 … 4e-9**, tức chỉ
là **nhiễu làm tròn số thực**. `StandardScaler` chia cho `sqrt(var)` → biến nhiễu đó thành tín
hiệu biên độ đơn vị:

| Đặc trưng | var | Khuếch đại |
|-----------|-----|------------|
| `spec.temp.centroid` | 1.15e-10 | **93.072×** |
| `spec.temp.band_mid` | 1.70e-10 | 76.739× |
| `spec.temp.band_high` | 2.54e-10 | 62.733× |
| `spec.temp.band_low` | 8.26e-10 | 34.788× |
| `spec.temp.gini` | 1.40e-09 | 26.727× |
| `stat.temp.waveform` | 2.60e-09 | 19.610× |
| `spec.temp.flatness` | 4.32e-09 | 15.210× |

Gần như trọn khối **phổ nhiệt độ**. Severson chạy trong buồng 30 °C nên nhiệt độ gần như không
đổi trong window 30 bước → phổ FFT là DC thuần → mọi mô tả hình dạng phổ suy biến.

7 kênh nhiễu này vào thẳng `film_proj` — mạng sinh `gamma`/`beta` **điều biến MỌI hidden unit**
— nên nhiễu lan ra toàn bộ biểu diễn và biểu hiện thành phương sai dự đoán, tức **RMSE**.

Fix: `FEATURE_VAR_FLOOR = 1e-8`, ép `scale_ = 1.0` cho các đặc trưng dưới ngưỡng → giá trị co về
~0 (input chết) thay vì input nhiễu. Ngưỡng nằm trong khoảng trống rõ rệt: khối gây hại ở
`var ~1e-10`, còn đặc trưng nhỏ nhất mang tín hiệu thật (`spec.voltage.centroid`) ở `~1.4e-6`.
**Không phải sửa code inference** — `inference.py` gọi `feature_scaler.transform()` trên chính
file pickle này nên tự động khớp.

Output: `soh_mamba_v2.0-lfp.pth`, `isolation_forest_v2.0-lfp.pkl`, `scaler_lfp.pkl`,
`feature_scaler_lfp.pkl` — 4 file riêng, không ghi đè artifact NASA.

## 1 — GPU check

In [ ]:
!nvidia-smi
import torch
print('PyTorch:', torch.__version__, '| CUDA:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Chua bat GPU: Settings -> Accelerator -> GPU T4 x2'
name = torch.cuda.get_device_name(0)
print('GPU:', name)
assert 'P100' not in name, 'P100 khong tuong thich PyTorch Kaggle (sm_60) - doi sang GPU T4 x2'

## 2 — Clone repo

In [ ]:
import subprocess, os
BRANCH = 'feat/GH-67-lfp-retrain-severson'
REPO   = '/kaggle/working/ai-module'
url = 'https://github.com/GSU26SE55/ai-module.git'
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret('GITHUB_TOKEN')
    url = f'https://{token}@github.com/GSU26SE55/ai-module.git'
except Exception as e:
    print('Khong co GITHUB_TOKEN secret -> thu public clone:', e)

if not os.path.isdir(REPO):
    subprocess.run(['git', 'clone', '--depth', '1', '-b', BRANCH, url, REPO], check=True)
os.chdir(REPO)
print(subprocess.check_output(['git', 'log', '-1', '--format=%h %ci %s']).decode())

## 3 — Kiểm tra code clone về có đúng bản mới không

Chặn đúng cái bẫy đã làm mất ~11 giờ: notebook đã sửa nhưng code trên GitHub thì chưa.
Nếu cell này fail → về máy chạy `git push` rồi **Restart & Run All** (phải xoá session để
clone lại, vì cell 2 skip clone khi thư mục đã tồn tại).

In [ ]:
import pathlib

checks = {
    'scripts/preprocess_lfp.py': ['--cycle-stride', 'PHYSICAL_RANGES', '_nonphysical_channel',
                                  '--phase', '_longest_discharge_segment',
                                  'TIME_UNIT_SECONDS', '--time-unit', 'FEATURE_VAR_FLOOR'],
    'scripts/train.py':          ['--feature-scaler-version', '--mamba-out', '--iso-out'],
    'src/core/config.py':        ['LFP_CYCLE_COUNT_NORM', 'LFP_NOMINAL_CAPACITY_AH'],
}
missing = []
for path, needles in checks.items():
    text = pathlib.Path(path).read_text(encoding='utf-8')
    for n in needles:
        status = 'OK  ' if n in text else 'THIEU'
        print(f'  [{status}] {path}: {n}')
        if n not in text:
            missing.append(f'{path}::{n}')

assert not missing, (
    'Code clone ve THIEU cac fix sau: ' + ', '.join(missing) +
    '\n-> Ve may chay: git add -A && git commit && git push, roi Restart & Run All.'
)
print('\nTat ca fix da co trong code clone ve.')

## 4 — Dependencies

In [ ]:
%pip install -q h5py scipy scikit-learn joblib pandas
import h5py, scipy, sklearn
print('h5py', h5py.__version__, '| scipy', scipy.__version__, '| sklearn', sklearn.__version__)

## 5 — Tìm dataset Severson

Cần ít nhất 1 file `*.mat` có chữ "batch" trong tên (vd
`2017-05-12_batchdata_updated_struct_errorcorrect.mat`). Fail → kiểm tra lại **+ Add Data**.

In [ ]:
import os, subprocess
found = [f for f in subprocess.check_output(
    ['find', '/kaggle/input', '-iname', '*batch*.mat']).decode().splitlines() if f]
assert found, 'Khong thay file *batch*.mat - dung + Add Data de attach dataset Severson truoc'
DATASET = os.path.dirname(found[0])
os.chdir('/kaggle/working/ai-module')
print('DATASET:', DATASET)
for f in found:
    print('  ', f)

## 6 — Preprocess (Severson `.mat` → window=30, 6 feature)

Hai fix **đúng đắn** (không phải tuning) áp dụng ở bước này, mặc định đã bật:

- **`--phase discharge`** — chỉ lấy đoạn **xả** dài nhất của mỗi cycle. Severson lưu nguyên
  cycle gồm cả sạc nhanh nhiều bước (dòng dương tới ~8 A) lẫn xả 4C, trong khi NASA
  (`scripts/preprocess.py`) chỉ nạp chu kỳ xả, và inference cũng chỉ thấy telemetry xả.
  Trước fix này ~nửa số window là pha **sạc** — model phải đoán nhãn dung lượng *xả*
  (`QDischarge`) từ mẫu sạc.
- **`PHYSICAL_RANGES`** — drop cycle chứa giá trị sensor phi vật lý.

### `--cycle-stride 5` là BẮT BUỘC, không phải tuỳ chọn

Số đo thật từ log lần 2 (full data, không cắt):

| Bước | Thời gian |
|------|-----------|
| Parse 4 file `.mat` | 4,5 phút |
| Trích xuất window + feature (train) | **3,7 giờ** |
| Trích xuất val/test | 24 phút |
| Train **5 epoch** trên 3 220 853 window | **7,0 giờ** (≈ **1,4 giờ/epoch**) |
| **Tổng** | **11,2 giờ** (giới hạn Kaggle: 12h) |

→ 100 epoch trên full data = **140 giờ**, vượt giới hạn **12 lần**. Không có cách nào chạy.

`--cycle-stride 5` + lọc pha xả giảm còn ~290k window → ~450 giây/epoch → 50 epoch ≈ 6,3 giờ,
cộng preprocess ~27 phút → **tổng ~7 giờ**, vừa khít. SOH giữa các cycle liền nhau của Severson
chênh ~0,01% nên bỏ 4/5 cycle gần như không mất thông tin.

### Đọc 4 thứ trong log

1. **`scaler range time`** — lần 3 ra `[0.000, 24.130]` (phút). Lần này phải ra khoảng
   **`[0.000, ~1450]`** (giây). Nếu vẫn ~24 → fix `--time-unit` chưa ăn.
2. **`scaler range temperature`** — kỳ vọng ~`[0, 45]`, không còn `[-270, 400]`.
3. **`scaler range current`** — phải **toàn số âm** (~`[-4.3, -0.1]`) = thuần pha xả.
4. **`discharge duration trung vi`** — sau fix phải in `giay (khop NASA)`, không phải `PHUT?`.
5. **`N/57 feature suy bien ... -> ep scale_=1.0`** — kỳ vọng ~7–12 feature (bug #8). Nếu in ra
   `0` thì fix `FEATURE_VAR_FLOOR` chưa ăn.
6. **Số cycle bị drop** — vài chục là bình thường. Drop tỉ lệ lớn → đang vứt dữ liệu thật.

Ngoài ra `[TIMING]` tách thời gian parse `.mat` vs trích xuất window/feature + số step/epoch.

In [ ]:
import os; os.chdir('/kaggle/working/ai-module')
!python scripts/preprocess_lfp.py --data-dir "{DATASET}" --output-dir data/processed_lfp \
    --cycle-stride 5 --phase discharge --time-unit minutes

## 7 — Train

Kiến trúc + hyperparameter **giữ nguyên** như model NASA. `--mamba-out`/`--iso-out`/
`--model-version` đảm bảo KHÔNG ghi đè `soh_mamba_v1.6.pth` production.

- **`--balance-bands`** (giữ từ lần 3) — trọng số loss tỉ lệ nghịch tần suất ô (nhiệt độ × SOH),
  kéo dải EOL thưa mẫu lên ngang dải 90–100% dày mẫu (trần 5×).
- **`--swa` (mới)** — trung bình hoá trọng số 25% epoch cuối. Rủi ro rất thấp: `train.py` chỉ
  dùng trọng số SWA **nếu val_loss của nó tốt hơn** checkpoint tốt nhất, không thì tự revert.
  Không tốn thêm epoch.

`--epochs 50`: với ~290k window, 50 epoch cho `ReduceLROnPlateau` ~10 lần cơ hội giảm LR và
early-stopping (`patience=15`) có chỗ hoạt động — điều chưa từng xảy ra ở lần 1/2 (5 epoch).

`train.py` chỉ in dòng metric khi `epoch % 10 == 0` → sẽ có 5 dòng. So `TrainLoss` với `ValLoss`:

- cả 2 còn cao và đang giảm → vẫn underfit, tăng epoch
- TrainLoss thấp mà ValLoss cao → overfit, thêm `--jitter 0.01`
- cả 2 phẳng sớm → đã hội tụ

Tìm thêm dòng `SWA: averaged N epochs | SWA val_loss=... vs best-ckpt val_loss=...` để biết SWA
có được dùng hay bị revert.

### Còn lại cho lần sau nếu vẫn chưa dưới 1%

`train()` cho window=30 chỉ nhận 4 knob: `epochs`, `--balance-bands`, `--jitter`, `--swa`
(mọi flag khác như `--pooling`/`--patch-size`/`--dropout` **chỉ dùng cho `--long`**, vô tác dụng
ở đây). Lần này dùng hết 3 knob an toàn. Còn lại:

1. `--cycle-stride 2` — gấp 2.5 lần dữ liệu, phải giảm còn ~25 epoch để vừa 12h
2. `--jitter 0.01` — nếu log cho thấy overfit
3. Tăng `d_model` 64→128 — **lệch spec `CLAUDE.md`**, cần quyết định riêng

In [ ]:
import os; os.chdir('/kaggle/working/ai-module')
!python scripts/train.py \
    --data-dir data/processed_lfp \
    --epochs 50 \
    --balance-bands \
    --swa \
    --log-dir logs/training \
    --mamba-out models/weights/soh_mamba_v2.0-lfp.pth \
    --iso-out models/weights/isolation_forest_v2.0-lfp.pkl \
    --model-version 2.0-lfp \
    --feature-scaler-version 2.0-lfp

## 8 — Kiểm tra kết quả

**Nhìn `Per-band` trước, MAE tổng sau.** MAE tổng bị dải 90–100% (88% số mẫu) chi phối nên
luôn trông đẹp. Chỉ số quyết định là **bias ở dải 70–80%**: lần 2 là **+10.280%** (pin 75% bị
báo thành 85% → bỏ sót pin cần thay). Lần này phải thấy con số đó giảm mạnh.

In [ ]:
import glob, os, re
import torch, joblib

PREV = {'mae': 1.4365, 'rmse': 1.8767, 'bias_70_80': +10.280}  # mae/rmse=lan 3, bias=lan 2
ck = torch.load('models/weights/soh_mamba_v2.0-lfp.pth', map_location='cpu', weights_only=False)
mae, rmse = ck['test_mae'], ck['test_rmse']

print('=== PER-BAND (quan trong nhat) ===')
logs = sorted(glob.glob('logs/training/train_*.log'), key=os.path.getmtime)
band_lines = []
if logs:
    for ln in open(logs[-1], encoding='utf-8'):
        if re.search(r'SOH\s+\d+-\d+', ln):
            band_lines.append(ln.rstrip())
            print('  ', ln.split('INFO')[-1].strip())
if not band_lines:
    print('   (khong doc duoc log - xem truc tiep output cell 7)')
else:
    # Tim dung DONG cua dai 70-80 roi moi bat bias — join cac dong lai va dung
    # `.*` se tham lam, vot nham bias cua dong cuoi cung.
    hit = next((l for l in band_lines if re.search(r'SOH\s*70-80', l)), None)
    m = re.search(r'bias=([+-][\d.]+)', hit) if hit else None
    if m:
        now = float(m.group(1))
        print(f"\n   bias dai 70-80%: {now:+.3f}%  (lan 2: {PREV['bias_70_80']:+.3f}%)"
              f"  -> cai thien {abs(PREV['bias_70_80']) - abs(now):+.3f} diem")
    else:
        print('\n   (dai 70-80% khong co mau trong test set lan nay)')

print('\n=== METRIC TONG ===')
print(f"MAE : {mae:.4f}%  (target <2.0, lan 3 {PREV['mae']})  -> {PREV['mae'] - mae:+.4f}")
print(f"RMSE: {rmse:.4f}%  (target <3.0, lan 3 {PREV['rmse']})  -> {PREV['rmse'] - rmse:+.4f}")
print('Dat target:', mae < 2.0 and rmse < 3.0)
print('balance_bands:', ck.get('balance_bands'), '| jitter:', ck.get('jitter'), '| swa:', ck.get('swa'))

print('\n=== XAC NHAN FIX PREPROCESS DA AN ===')
s = joblib.load('models/weights/scaler_lfp.pkl'); sc = s['scaler']
for i, n in enumerate(['voltage', 'current', 'temperature', 'time']):
    print(f'  {n:<12}: [{sc.data_min_[i]:10.3f}, {sc.data_max_[i]:10.3f}]')
print('  metadata:', {k: v for k, v in s.items() if k not in ('scaler', 'trained_on')})
assert s.get('phase') == 'discharge', 'scaler KHONG co phase=discharge -> chay bang code CU!'
assert sc.data_max_[1] <= 0, 'current phai toan AM (thuan pha xa)'
assert sc.data_max_[3] > 200, (
    f'time max = {sc.data_max_[3]:.1f} -> van la PHUT, fix --time-unit chua an. '
    'Ky vong ~1450 (giay).'
)
fs = joblib.load('models/weights/feature_scaler_lfp.pkl')
print(f"  feature_scaler: var_floor={fs.get('var_floor')} "
      f"n_degenerate={fs.get('n_degenerate')}")
assert fs.get('n_degenerate') is not None, 'feature_scaler thieu n_degenerate -> code CU'
amp_max = float((1.0 / fs['scaler'].scale_).max())
print(f'  khuech dai lon nhat con lai: {amp_max:,.0f}x (lan 3: 93,072x)')
assert amp_max < 5000, f'van con feature khuech dai {amp_max:,.0f}x -> fix #8 chua an'
print('  => scaler sinh boi code moi: phase=discharge, current<0, time GIAY, het nhieu feature')

## 9 — Đóng gói artifact để tải về

4 file cần copy vào `models/weights/` trên máy. **Không tự commit** — tải zip về, tự
`git add` + `git commit` + `git push` theo quy trình repo.

In [ ]:
import shutil, os
OUT = '/kaggle/working/lfp_artifacts'
os.makedirs(OUT, exist_ok=True)
for p in [
    'models/weights/soh_mamba_v2.0-lfp.pth',
    'models/weights/isolation_forest_v2.0-lfp.pkl',
    'models/weights/scaler_lfp.pkl',
    'models/weights/feature_scaler_lfp.pkl',
]:
    shutil.copy(p, OUT)
    print('  +', os.path.basename(p), f'({os.path.getsize(p)/1024:.0f} KB)')
shutil.make_archive(OUT, 'zip', OUT)
print('\nTai ve: lfp_artifacts.zip (tab Output ben phai)')

## 10 — Dọn output (xoá repo clone — artifact đã nằm trong zip)

In [ ]:
import os, shutil
os.chdir('/kaggle/working')
shutil.rmtree('/kaggle/working/ai-module', ignore_errors=True)
shutil.rmtree('/kaggle/working/lfp_artifacts', ignore_errors=True)
print('Output con lai:', sorted(os.listdir('/kaggle/working')))